# Random forest: pre-bid risk

Run `python preprocessing.py` once from the project folder. The input has one row per auction, and IsBadBuy = 1 is the positive class. Compare the two objectives on the same validation cars using continuous scores.

In [ ]:
import pandas as pd
from preprocessing import PRICE_RESULTS, VALIDATION_RESULTS, load_development
from evaluation import save_validation

# Selected 15-field configuration; use False only to reproduce the 11-field baseline.
include_prices = True
X_train, y_train, X_val, y_val = load_development(include_prices=include_prices)
result_dir = VALIDATION_RESULTS
if include_prices:
    result_dir = PRICE_RESULTS
scores = pd.DataFrame(index=y_val.index)
print(f"{len(X_train):,} training cars, {len(X_val):,} validation cars")

### Train-only encoding
Fit one shared encoder for the two forests. An unseen category is ignored by the encoder.

In [ ]:
from tree_models import BASELINE, encode_forest, make_tree

train_encoded, val_encoded = encode_forest(X_train, X_val)

### Matched forest settings
Each tree uses squared error or log loss to choose splits. Both forests use the same sampling and feature budget.

In [ ]:
depth, min_leaf = BASELINE["random_forest"]
rf_l2 = make_tree("RF_L2", depth, min_leaf)
rf_l2.fit(train_encoded, y_train)
scores["RF_L2"] = rf_l2.predict(val_encoded)

rf_log = make_tree("RF_Logloss", depth, min_leaf)
rf_log.fit(train_encoded, y_train)
scores["RF_Logloss"] = rf_log.predict_proba(val_encoded)[:, 1]

### Validation AUC
Scores and labels are saved by RefId for a paired comparison with CatBoost.

In [ ]:
save_validation(y_val, scores, result_dir, "random_forest")

### Hyperparameter tuning

Run `%run tune_trees.py random_forest` in this notebook to screen the six settings for each objective and refit the finalists. After both families finish, run `%run tune_trees.py summarize` for the validation AUC comparison and bidding thresholds.